# Restore original ElevenLabs audio to Google Drive
One-click Colab recovery of 58 existing original audio recordings, without creating new speech. Requires access to the private recovery manifest in your own Google Drive. Run all cells and authorise Google Drive.
The private manifest includes 58 refreshed links and stable history IDs. If links expire, add your ElevenLabs API key in Colab Secrets as `ELEVENLABS_API_KEY` to retrieve existing originals through the ElevenLabs history audio endpoint (not generate new speech).


In [ ]:
import csv, hashlib, json, re, time
from pathlib import Path
import requests
from google.colab import auth, drive
import google.auth
from googleapiclient.discovery import build

print("Authorize your Google Drive to recover previously generated MP3 files. No audio regeneration.")
auth.authenticate_user()
drive.mount("/content/drive")
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
api = build("drive","v3",credentials=creds,cache_discovery=False)

title = "Vivian_Shilpa_ElevenLabs_58_Original_Takes_Recovery_Manifest_2026-10-25"
query = "name = '" + title + "' and mimeType = 'application/vnd.google-apps.document' and trashed = false"
matches=api.files().list(q=query,fields="files(id,name)",page_size=100).execute().get("files",[])
assert len(matches)==1, f"Expected one private recovery document in this Google account, got {len(matches)}"
text = api.files().export(fileId=matches[0]["id"],mimeType="text/plain").execute().decode("utf-8-sig")
entries = [json.loads(line) for line in text.splitlines() if line.strip().startswith('{"index":')]
refreshed = [json.loads(line) for line in text.splitlines() if line.strip().startswith('{"generation_id":')]
latest = {item["generation_id"]:item for item in refreshed}
for rec in entries:
 rec.update(latest.get(rec["generation_id"], {}))
assert len(entries)==58, f"Expected 58 original takes, found {len(entries)}"
print(f"Signed URL records: {len(entries)}; durable history IDs: {sum(bool(x.get('history_item_id')) for x in entries)}")
dest=Path("/content/drive/MyDrive/ParrisVoiceBank/Vivian_Shilpa_Original_ElevenLabs_Recordings_2026-10-25")
dest.mkdir(parents=True,exist_ok=True)
existing={}
for p in dest.glob("*.mp3"):
 try: existing[hashlib.sha256(p.read_bytes()).hexdigest()]=p.name
 except: pass
voice_per_node=["Vivian","Shilpa","Vivian","Vivian","Vivian","Shilpa","Shilpa","Shilpa","Vivian","Vivian","Vivian","Shilpa","Vivian","Shilpa","Shilpa","Vivian","Shilpa","Vivian","Vivian","Vivian","Shilpa","Shilpa","Vivian","Shilpa","Vivian","Vivian","Shilpa","Shilpa","Vivian"]
audit=[]
for rec in entries:
 n=int(rec["index"])
 gid=rec["generation_id"]
 voice=voice_per_node[(n-1)//2]
 slug=re.sub(r"[^a-z0-9]+","_",rec["prompt"].lower()).strip("_")[:42]
 name=f"{voice}_{n:02d}_{slug}_{gid}.mp3"
 file=dest/name
 status="error"; matching=""; sha="";error=""
 try:
  if file.exists() and file.stat().st_size>100:
   data=file.read_bytes()
  else:
   try:
    response=requests.get(rec["content_url"],timeout=60)
    response.raise_for_status()
    data=response.content
    if len(data)<200 or not (data.startswith(b"ID3") or (len(data)>1 and data[0]==255 and data[1]&224==224)):
     raise ValueError("Signed URL returned something other than MP3")
   except Exception as signed_error:
    history_id=rec.get("history_item_id")
    if not history_id:
     raise signed_error
    from google.colab import userdata
    try: key=userdata.get("ELEVENLABS_API_KEY")
    except Exception: key=None
    if not key:
     raise RuntimeError("Original signed URL expired, and ELEVENLABS_API_KEY is not set in Colab Secrets")
    api_response=requests.get("https://api.elevenlabs.io/v1/history/"+history_id+"/audio",headers={"xi-api-key":key},timeout=90)
    api_response.raise_for_status()
    data=api_response.content
  if len(data)<200 or not (data.startswith(b"ID3") or (len(data)>1 and data[0]==255 and data[1]&224==224)):
   raise ValueError("Downloaded response does not contain MP3 audio")
  sha=hashlib.sha256(data).hexdigest()
  if sha in existing:
   status="already_in_drive";matching=existing[sha]
  else:
   file.write_bytes(data)
   existing[sha]=file.name
   status="downloaded"; matching=file.name
 except Exception as exc:
  error=str(exc)[:250]
 audit.append({"index":n,"generation_id":gid,"voice":voice,"duration":rec.get("duration_secs"),"status":status,"filename":matching,"sha256":sha,"error":error})
 if n%10==0 or n==58: print(n,"/",58,"processed",sum(x["status"]!="error" for x in audit),"matched or saved")
 time.sleep(.1)
with (dest/"ElevenLabs_original_58_take_audit.csv").open("w",newline="",encoding="utf8") as f:
 writer=csv.DictWriter(f,fieldnames=list(audit[0]))
 writer.writeheader()
 writer.writerows(audit)
good=sum(x["status"]!="error" for x in audit)
print("BACKUP RESULT:",good,"/58 saved or exact-hash matched. Errors:",58-good)
if good!=58: print("Some temporary source links may have expired; the private recovery document retains original IDs.")


